# 05 — Computing Greeks

This notebook demonstrates how to compute option Greeks using every
available calculation method:

| Method | Engine | Greeks |
|---|---|---|
| `ANALYTICAL` | BSM | delta, gamma, vega, theta, rho |
| `TREE` | Binomial | delta, gamma, theta |
| `GRID` | PDE FD | delta, gamma, theta |
| `NUMERICAL` | Any | all (bump-and-revalue) |
| `PATHWISE` | MC | delta, gamma, vega, theta |
| `LIKELIHOOD_RATIO` | MC | delta, vega, theta |

The `greek_calc_method` argument on each Greek method controls the
calculation approach. When omitted, a sensible default is chosen
automatically.

## 1) Setup

In [1]:
import datetime as dt

import portfolio_analytics as pa

In [2]:
pricing_date = dt.datetime(2025, 6, 15)
maturity = dt.datetime(2026, 6, 15)
r = 0.05

curve = pa.DiscountCurve.flat(rate=r, end_time=2.0)
market = pa.MarketData(pricing_date=pricing_date, discount_curve=curve, currency="USD")
underlying = pa.UnderlyingData(initial_value=100.0, volatility=0.20, market_data=market)

spec = pa.VanillaSpec(
    option_type=pa.OptionType.CALL,
    exercise_type=pa.ExerciseType.EUROPEAN,
    strike=100.0,
    maturity=maturity,
)

## 2) BSM Greeks — Analytical (Exact)

BSM Greeks default to `ANALYTICAL`, which uses the closed-form BSM formulas.

In [3]:
bsm = pa.OptionValuation(underlying=underlying, spec=spec, pricing_method=pa.PricingMethod.BSM)

print("BSM Greeks (analytical):")
print(f"  Price: {bsm.present_value():.6f}")
print(f"  Delta: {bsm.delta():.6f}")
print(f"  Gamma: {bsm.gamma():.6f}")
print(f"  Vega:  {bsm.vega():.6f}")
print(f"  Theta: {bsm.theta():.6f}")
print(f"  Rho:   {bsm.rho():.6f}")

BSM Greeks (analytical):
  Price: 10.450584
  Delta: 0.636831
  Gamma: 0.018762
  Vega:  0.375240
  Theta: -0.017573
  Rho:   0.532325


You can also request numerical (bump-and-revalue) Greeks from BSM to
validate the analytical formulas:

In [4]:
NUMERICAL = pa.GreekCalculationMethod.NUMERICAL

print("BSM Greeks (numerical):")
print(f"  Delta: {bsm.delta(greek_calc_method=NUMERICAL):.6f}")
print(f"  Gamma: {bsm.gamma(greek_calc_method=NUMERICAL):.6f}")
print(f"  Vega:  {bsm.vega(greek_calc_method=NUMERICAL):.6f}")
print(f"  Theta: {bsm.theta(greek_calc_method=NUMERICAL):.6f}")
print(f"  Rho:   {bsm.rho(greek_calc_method=NUMERICAL):.6f}")

BSM Greeks (numerical):
  Delta: 0.636745
  Gamma: 0.018760
  Vega:  0.375210
  Theta: -0.017581
  Rho:   0.532306


## 3) Binomial Tree Greeks

Binomial trees provide `TREE` Greeks (from the first few tree layers) and
`NUMERICAL` Greeks (bump-and-revalue).

In [5]:
TREE = pa.GreekCalculationMethod.TREE

binom = pa.OptionValuation(
    underlying=underlying,
    spec=spec,
    pricing_method=pa.PricingMethod.BINOMIAL,
    params=pa.BinomialParams(num_steps=500),
)

print("Binomial Greeks (tree):")
print(f"  Delta: {binom.delta(greek_calc_method=TREE):.6f}")
print(f"  Gamma: {binom.gamma(greek_calc_method=TREE):.6f}")
print(f"  Theta: {binom.theta(greek_calc_method=TREE):.6f}")

print("\nBinomial Greeks (numerical):")
print(f"  Vega:  {binom.vega(greek_calc_method=NUMERICAL):.6f}")
print(f"  Rho:   {binom.rho(greek_calc_method=NUMERICAL):.6f}")

Binomial Greeks (tree):
  Delta: 0.636767
  Gamma: 0.018794
  Theta: -0.017590

Binomial Greeks (numerical):
  Vega:  0.375022
  Rho:   0.532282


## 4) PDE Grid Greeks

PDE finite difference provides `GRID` Greeks, extracted directly from the
finite difference grid at time zero. These are typically more accurate than
numerical bump-and-revalue for delta, gamma, and theta.

In [6]:
GRID = pa.GreekCalculationMethod.GRID

pde = pa.OptionValuation(
    underlying=underlying,
    spec=spec,
    pricing_method=pa.PricingMethod.PDE_FD,
    params=pa.PDEParams(),
)

print("PDE Greeks (grid):")
print(f"  Delta: {pde.delta(greek_calc_method=GRID):.6f}")
print(f"  Gamma: {pde.gamma(greek_calc_method=GRID):.6f}")
print(f"  Theta: {pde.theta(greek_calc_method=GRID):.6f}")

print("\nPDE Greeks (numerical):")
print(f"  Vega:  {pde.vega(greek_calc_method=NUMERICAL):.6f}")
print(f"  Rho:   {pde.rho(greek_calc_method=NUMERICAL):.6f}")

PDE Greeks (grid):


  Delta: 0.636580
  Gamma: 0.018791
  Theta: -0.017601

PDE Greeks (numerical):
  Vega:  0.375750
  Rho:   0.532241


## 5) Monte Carlo Greeks

MC supports three Greek calculation methods:

- **`NUMERICAL`** — bump-and-revalue (works for all Greeks, but requires
  re-simulation)
- **`PATHWISE`** — differentiates through the payoff (delta, vega)
- **`LIKELIHOOD_RATIO`** — score function method (delta, vega)

**Note:** MC Greeks are inherently noisy. Pathwise delta and vega tend to
have lower variance than likelihood ratio. Numerical theta via MC requires
re-simulation with a shifted time grid, so it can deviate more from BSM
than other Greeks.

In [11]:
PATHWISE = pa.GreekCalculationMethod.PATHWISE
LR = pa.GreekCalculationMethod.LIKELIHOOD_RATIO

sim_config = pa.SimulationConfig(paths=100_000, end_date=maturity, num_steps=100)
gbm = pa.GBMProcess(
    market_data=market,
    process_params=pa.GBMParams(initial_value=100.0, volatility=0.20),
    sim_config=sim_config,
)

mc = pa.OptionValuation(
    underlying=gbm,
    spec=spec,
    pricing_method=pa.PricingMethod.MONTE_CARLO,
    params=pa.MonteCarloParams(random_seed=42),
)

print("MC Greeks (pathwise):")
print(f"  Delta: {mc.delta(greek_calc_method=PATHWISE):.6f}")
print(f"  Gamma: {mc.gamma(greek_calc_method=PATHWISE):.6f}")
print(f"  Vega:  {mc.vega(greek_calc_method=PATHWISE):.6f}")
print(f"  Theta:  {mc.theta(greek_calc_method=PATHWISE):.6f}")

# Likelihood ratio has higher variance, especially for vega
print("\nMC Greeks (likelihood ratio):")
print(f"  Delta: {mc.delta(greek_calc_method=LR):.6f}")
print(f"  Vega:  {mc.vega(greek_calc_method=LR):.6f}")
print(f"  Theta:  {mc.theta(greek_calc_method=LR):.6f}")

print("\nMC Greeks (numerical bump-and-revalue):")
print(f"  Delta: {mc.delta(greek_calc_method=NUMERICAL):.6f}")
print(f"  Gamma: {mc.gamma(greek_calc_method=NUMERICAL):.6f}")
print(f"  Vega:  {mc.vega(greek_calc_method=NUMERICAL):.6f}")
print(f"  Theta: {mc.theta(greek_calc_method=NUMERICAL):.6f}")
print(f"  Rho:   {mc.rho(greek_calc_method=NUMERICAL):.6f}")

MC Greeks (pathwise):
  Delta: 0.637997
  Gamma: 0.018963
  Vega:  0.379429
  Theta:  -0.017694

MC Greeks (likelihood ratio):
  Delta: 0.647302
  Vega:  0.398246
  Theta:  -0.018337

MC Greeks (numerical bump-and-revalue):
  Delta: 0.637519
  Gamma: 0.019041
  Vega:  0.379443
  Theta: -0.017702
  Rho:   0.532571


## 6) Custom Bump Size

The `epsilon` parameter on `delta()` and `gamma()` controls the spot bump
size for numerical Greeks (as a fraction of spot).

In [8]:
print("BSM numerical delta with different epsilon:")
for eps in [0.001, 0.005, 0.01, 0.05]:
    d = bsm.delta(epsilon=eps, greek_calc_method=NUMERICAL)
    print(f"  epsilon={eps:.3f}: delta={d:.6f}")

print(f"\n  Analytical:       delta={bsm.delta():.6f}")

BSM numerical delta with different epsilon:
  epsilon=0.001: delta=0.636831
  epsilon=0.005: delta=0.636831
  epsilon=0.010: delta=0.636831
  epsilon=0.050: delta=0.636830

  Analytical:       delta=0.636831


## 7) Greeks for American Options

American options only support `TREE`/`GRID`/`NUMERICAL` Greeks (no closed-form).

In [9]:
am_put = pa.VanillaSpec(
    option_type=pa.OptionType.PUT,
    exercise_type=pa.ExerciseType.AMERICAN,
    strike=100.0,
    maturity=maturity,
)

binom_am = pa.OptionValuation(
    underlying=underlying,
    spec=am_put,
    pricing_method=pa.PricingMethod.BINOMIAL,
    params=pa.BinomialParams(num_steps=500),
)

print("American put Greeks (binomial tree):")
print(f"  Delta: {binom_am.delta(greek_calc_method=TREE):.6f}")
print(f"  Gamma: {binom_am.gamma(greek_calc_method=TREE):.6f}")
print(f"  Theta: {binom_am.theta(greek_calc_method=TREE):.6f}")
print(f"  Vega:  {binom_am.vega():.6f}")  # defaults to NUMERICAL
print(f"  Rho:   {binom_am.rho():.6f}")  # defaults to NUMERICAL

American put Greeks (binomial tree):
  Delta: -0.411170
  Gamma: 0.023018
  Theta: -0.006144
  Vega:  0.374753
  Rho:   -0.302189


## 8) Cross-Method Comparison

Compare delta across all methods for the same contract.

In [10]:
analytical_delta = bsm.delta()

print(f"{'Method':<30} {'Delta':>8} {'Diff':>10}")
print("-" * 50)
for label, d in [
    ("BSM (analytical)", bsm.delta()),
    ("BSM (numerical)", bsm.delta(greek_calc_method=NUMERICAL)),
    ("Binomial (tree)", binom.delta(greek_calc_method=TREE)),
    ("Binomial (numerical)", binom.delta(greek_calc_method=NUMERICAL)),
    ("PDE (grid)", pde.delta(greek_calc_method=GRID)),
    ("PDE (numerical)", pde.delta(greek_calc_method=NUMERICAL)),
    ("MC (pathwise)", mc.delta(greek_calc_method=PATHWISE)),
    ("MC (likelihood ratio)", mc.delta(greek_calc_method=LR)),
    ("MC (numerical)", mc.delta(greek_calc_method=NUMERICAL)),
]:
    print(f"{label:<30} {d:>8.6f} {d - analytical_delta:>+10.6f}")

Method                            Delta       Diff
--------------------------------------------------
BSM (analytical)               0.636831  +0.000000
BSM (numerical)                0.636745  -0.000086
Binomial (tree)                0.636767  -0.000064
Binomial (numerical)           0.636767  -0.000064
PDE (grid)                     0.636580  -0.000250
PDE (numerical)                0.636667  -0.000163
MC (pathwise)                  0.637997  +0.001166
MC (likelihood ratio)          0.647302  +0.010472
MC (numerical)                 0.637519  +0.000689
